# Week 52, Day 2: Review II: coding, decoding and real data

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w052/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: intervals and counts

Generate 2,000 s of spikes at 20 Hz from a Poisson process (exponential intervals) and from a gamma renewal process with shape 4 (same mean interval). For each, compute the CV of the intervals and the Fano factor of spike counts in 1 s windows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(34)
rate, T = 20.0, 2000.0
isis = {"poisson": r.exponential(1 / rate, int(rate * T * 1.2)),
        "gamma": r.gamma(4, 1 / (4 * rate), int(rate * T * 1.2))}
cv, ff = {}, {}
for name, isi in isis.items():
    t = np.cumsum(isi); t = t[t < T]
    cv[name] = ...
    counts = np.histogram(t, bins=np.arange(0, T + 1, 1.0))[0]
    ff[name] = ...
    print(f"{name:8s}: CV {cv[name]:.3f}, CV squared {cv[name] ** 2:.3f}, Fano factor {ff[name]:.3f}")

In [ ]:
import numpy as np
def test_cv():
    "cv holds the interval CV of the Poisson and gamma trains, near 1 and 0.5"
    assert abs(cv["poisson"] - 1) < 0.05 and abs(cv["gamma"] - 0.5) < 0.03
def test_ff():
    "ff holds the Fano factor of counts in 1 s windows, near CV squared"
    assert abs(ff["poisson"] - 1) < 0.1 and abs(ff["gamma"] - 0.25) < 0.05

test_cv()
test_ff()
print("All checks passed")

### Exercise: Problem 2: a GLM that finds refractoriness

Build the design matrix with the stimulus, the spike counts 1 to 5 bins back and a constant. Write poisson_newton(X, y), Newton's method for the Poisson GLM with log link, and compare the log-likelihood with and without the history columns.

In [ ]:
T2 = 20000
r = np.random.default_rng(35)
stim = r.normal(0, 1, T2)
w_true = np.array([0.8, -3.0, -1.5, -0.6, -0.2, 0.0, -2.5])   # stimulus, history lags 1 to 5, bias
y = np.zeros(T2)
for t in range(T2):
    past = [y[t - k] if t - k >= 0 else 0 for k in range(1, 6)]
    y[t] = r.poisson(np.exp(w_true[0] * stim[t] + np.dot(w_true[1:6], past) + w_true[6]))
X = ...

def poisson_newton(X, y, iters=20):
    w = np.zeros(X.shape[1])
    for _ in range(iters):
        lam = np.exp(X @ w)
        g = ...
        H = ...
        w -= np.linalg.solve(H, g)
    return w

def loglik(X, y, w):
    z = X @ w
    return np.sum(y * z - np.exp(z))

w_hat = poisson_newton(X, y)
w_stim = poisson_newton(X[:, [0, 6]], y)
ll_full, ll_stim = loglik(X, y, w_hat), loglik(X[:, [0, 6]], y, w_stim)
print("true:  ", np.round(w_true, 2)); print("fitted:", np.round(w_hat, 2))
print(f"log-likelihood gain from history: {ll_full - ll_stim:.0f}; spikes: {int(y.sum())}")

In [ ]:
import numpy as np
def test_design():
    "X has the stimulus, five lagged spike counts and a constant"
    assert X.shape == (T2, 7) and np.allclose(X[:, 2], np.r_[0, 0, y[:-2]]) and np.allclose(X[:, -1], 1)
def test_fit():
    "the fitted weights are within 0.3 of the true ones"
    assert np.abs(w_hat - w_true).max() < 0.3
def test_ll():
    "the history terms raise the log-likelihood by more than 100"
    assert ll_full - ll_stim > 100

test_design()
test_fit()
test_ll()
print("All checks passed")

### Exercise: Problem 3: a decoder against its bound

Write fisher(s) for 40 Poisson neurons with Gaussian tuning observed for Tw seconds (differentiate the tuning curves analytically or numerically). Then decode 2,000 simulated trials at s0 = 10 degrees by maximum likelihood on a fine grid, and compare the variance of the estimates with 1/fisher(s0).

In [ ]:
r = np.random.default_rng(36)
pref = np.linspace(-90, 90, 40); width, peak, base, Tw = 20.0, 20.0, 2.0, 0.5

def tuning(s):
    return base + peak * np.exp(-0.5 * ((s - pref) / width) ** 2)

def fisher(s):
    f = tuning(s)
    fprime = ...
    return ...

s0, grid = 10.0, np.linspace(-20, 40, 1201)
F = np.array([tuning(g) for g in grid])            # grid x neurons
counts = r.poisson(tuning(s0) * Tw, (2000, 40))
loglik_grid = counts @ np.log(F * Tw).T - (F * Tw).sum(1)
est = ...
print(f"Fisher information at {s0} degrees: {fisher(s0):.3f} per degree squared; bound on SD {1 / np.sqrt(fisher(s0)):.2f} degrees")
print(f"ML decoder: mean {est.mean():.2f}, SD {est.std():.2f}; variance times Fisher information {est.var() * fisher(s0):.2f}")

In [ ]:
import numpy as np
def test_fi():
    "fisher matches T times the sum of f' squared over f"
    s = 10.0; fp = (tuning(s + 1e-4) - tuning(s - 1e-4)) / 2e-4
    assert abs(fisher(s) - Tw * np.sum(fp ** 2 / tuning(s))) / fisher(s) < 1e-4
def test_ml():
    "the decoder is nearly unbiased"
    assert abs(est.mean() - s0) < 0.3
def test_bound():
    "the decoder's variance is within 20% above the bound and never far below it"
    assert 0.9 < est.var() * fisher(s0) < 1.2

test_fi()
test_ml()
test_bound()
print("All checks passed")